In [ ]:
import json
import os
import sys
from typing import List, Optional, Dict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm

sys.path.append('..')
from plotting_utils import (
    load_in_combined_results,
    load_monitoring_data,
    plot_markov_analysis,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    plot_cost_vs_safety_curves,
    get_roc_curve,
    interpolate_tpr,
    prepare_monitoring_data,
    merge_monitoring_with_combined_results,
    plot_training_progress,
    plot_all_suspiciousness_metrics,
    is_in_training_data,
    filter_and_prepare_data,
    prepare_data_for_training_progress,
)
from utils.analysis_utils import extract_verdict
from utils.path_utils import resolve_path

# Constants
MAIN_AGENT_MODELS = ['claude-3-5-sonnet-20241022']

default_step_mapping = {
    "gpt-4o-2024-08-06": 0,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1:1:BA612ygq:ckpt-step-70": 70,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1:1:BA612NBM:ckpt-step-140": 140,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1:1:BA613xjZ": 210, 
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAM3F3vz:ckpt-step-280": 280,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAM3GipM": 350,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAQ9t3GI:ckpt-step-560": 560,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAQ9tmgK:ckpt-step-630": 630,
    "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAQ9tTfH": 700,
    }
default_step_mapping = {
    "gpt-4o-mini-2024-07-18": 0,
    "ft:gpt-4o-mini-2024-07-18:scale-safety-research-1:combined-3-reviewed:BF5TXVGA:ckpt-step-70": 70,
    "ft:gpt-4o-mini-2024-07-18:scale-safety-research-1:combined-3-reviewed:BF5TX0st:ckpt-step-140": 140,
    "ft:gpt-4o-mini-2024-07-18:scale-safety-research-1:combined-3-reviewed:BF5TXzWt": 210,
}
FT_MONITOR_MODELS = list(default_step_mapping.keys())
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022'] + FT_MONITOR_MODELS
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]

# Cache for combined_results
_combined_results_cache = {}
NUM_MONITORS = 3

def load_solution_reviews_data(base_dir: str = "monitor_responses/system_prompt_5_solution_reviews") -> pd.DataFrame:
    """
    Load all solution review JSON files from the specified directory structure.
    
    Directory structure:
    base_dir/
        task_name/
            model_name.json
    
    Returns:
        pd.DataFrame: DataFrame containing the loaded data with metadata
    """
    all_data = []
    base_path = resolve_path(base_dir)
    
    # Process each task directory
    for task_dir in base_path.iterdir():
        if not task_dir.is_dir():
            continue
        
        task_name = task_dir.name
        
        # Process each JSON file in the task directory
        for json_file in task_dir.glob('*.json'):
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                
                # Use the filename (without .json) as the model name
                model_name = json_file.stem
                
                # Add metadata about the file location
                entry = {
                    'task_name': task_name,
                    'model_name': model_name,
                    'file_name': json_file.name,
                    'data': data
                }
                all_data.append(entry)
            except Exception as e:
                print(f"Error loading {json_file}: {e}")
                continue
    
    # Convert to DataFrame
    if not all_data:
        print("No data found in the specified directory.")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_data)
    
    # Extract verdicts from responses if they exist
    for i in range(1, 6):
        response_key = f'response_{i}'
        df[f'verdict_{i}'] = df.apply(
            lambda row: extract_verdict(
                row['data'].get(response_key, {}).get('content', ''), 
                tag_word='verdict'
            ) if response_key in row['data'] else None,
            axis=1
        )
        
        # Extract usage information if available
        df[f'usage_{i}'] = df.apply(
            lambda row: row['data'].get(response_key, {}).get('usage', [{}])[0] 
                      if response_key in row['data'] and 
                         'usage' in row['data'].get(response_key, {}) and 
                         len(row['data'].get(response_key, {}).get('usage', [])) > 0 
                      else {},
            axis=1
        )
    
    # Compute aggregate metrics
    suspiciousness_cols = [f'verdict_{i}' for i in range(1, NUM_MONITORS + 1) if f'verdict_{i}' in df.columns]
    if suspiciousness_cols:
        df['max_verdict'] = df[suspiciousness_cols].max(axis=1)
        df['mean_verdict'] = df[suspiciousness_cols].mean(axis=1)
        df['median_verdict'] = df[suspiciousness_cols].median(axis=1)
        df['min_verdict'] = df[suspiciousness_cols].min(axis=1)
        df['std_verdict'] = df[suspiciousness_cols].std(axis=1)
    
    return df


In [ ]:
def load_combined_results(include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and cache combined_results."""
    cache_key = f"{include_failed_attempts}_{include_failed_main_task}"
    if cache_key in _combined_results_cache:
        print(f"Using cached combined_results for {cache_key}")
        return _combined_results_cache[cache_key]
    combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)
    _combined_results_cache[cache_key] = combined_results
    return combined_results

def prepare_system_prompt_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool) -> pd.DataFrame:
    """Load and prepare the initial datasets."""
    # Use the generalized function from plotting_utils
    system_prompt_df = prepare_monitoring_data(
        base_dir=base_dir,
        monitor_models=FT_MONITOR_MODELS,  # Filter to only include fine-tuned models
        num_monitors=NUM_MONITORS,  # Process all monitor responses
        compute_aggregates=True,  # Compute max, mean, etc.
        tag_word='verdict'
    )
    
    return system_prompt_df

def merge_and_compute_features_for_system_prompt(combined_results: pd.DataFrame, 
                             system_prompt_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    return merge_monitoring_with_combined_results(
        monitoring_df=system_prompt_df,
        combined_results=combined_results,
        how='inner',  # Use inner join as in the original
        compute_convo_length=True
    )


In [ ]:
def load_and_merge_all_system_prompts(include_failed_attempts=False, include_failed_main_task=False):
    """
    Load all system_prompt dataframes (1-9) and merge them on common columns.
    
    Args:
        include_failed_attempts: Whether to include failed attempts
        include_failed_main_task: Whether to include failed main tasks
        
    Returns:
        pd.DataFrame: Merged dataframe containing data from all system prompts
    """
    # Load all system prompt dataframes
    system_prompt_dfs = {}
    for i in [1, 5]:  # 1 to 9 inclusive
        base_dir = f'monitor_responses/system_prompt_{i}'
        system_prompt_df = prepare_system_prompt_data(base_dir=base_dir, 
                                       include_failed_attempts=include_failed_attempts, 
                                       include_failed_main_task=include_failed_main_task)
        combined_results = load_combined_results(include_failed_attempts=include_failed_attempts, 
                                       include_failed_main_task=include_failed_main_task)
        df = merge_and_compute_features_for_system_prompt(combined_results, system_prompt_df)
        # Drop the data column before merging to avoid duplicate column issues
        if 'data' in df.columns:
            df = df.drop(columns=['data', 'default_suspiciousness', 'main_task_evaluation', 'main_task_success', 'num_assistant_messages', 'num_failed_tool_calls', 'combined_score', 'convo_length'])
        system_prompt_dfs[i] = df
        print(f"Loaded system_prompt_{i} with shape {df.shape}")
    
    # Start with the first dataframe
    merged_df = system_prompt_dfs[1]
    
    # Merge with the rest of the dataframes
    for i in [5]:
        merged_df = pd.merge(
            merged_df,
            system_prompt_dfs[i],
            on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 'side_task_evaluation'],
            how='inner',
            suffixes=(f'_{i-1}', f'_{i}')  # Use explicit suffixes to avoid conflicts
        )
        print(f"After merging system_prompt_{i}, shape: {merged_df.shape}")
    
    return merged_df


In [ ]:
# Load combined results for use in later cells
combined_results = load_combined_results(include_failed_attempts=False, include_failed_main_task=False)

In [ ]:
system_prompt_df_1 = prepare_system_prompt_data(base_dir='monitor_responses/system_prompt_1_temp_0', include_failed_attempts=False, include_failed_main_task=False)
system_prompt_df_5 = prepare_system_prompt_data(base_dir='monitor_responses/system_prompt_5_temp_0', include_failed_attempts=False, include_failed_main_task=False)
system_prompt_df_1['side_task_evaluation'] = system_prompt_df_1.transcript_type != 'transcripts_no_secondary_task'
system_prompt_df_5['side_task_evaluation'] = system_prompt_df_5.transcript_type != 'transcripts_no_secondary_task'

training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
training_data = pd.read_csv(training_data_path)

system_prompt_df_1['in_training_data'] = system_prompt_df_1.apply(
    lambda row: is_in_training_data(row, training_data), 
    axis=1
)
system_prompt_df_5['in_training_data'] = system_prompt_df_5.apply(
    lambda row: is_in_training_data(row, training_data), 
    axis=1
)
both_system_prompts_df = pd.merge(
    system_prompt_df_1,
    system_prompt_df_5,
    on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name', 'in_training_data', 'side_task_evaluation'],
    suffixes=('_sp1', '_sp5'),
    how = 'inner'
)
both_system_prompts_df['total_max'] = both_system_prompts_df[['max_suspiciousness_sp1', 'max_suspiciousness_sp5']].max(axis=1)
both_system_prompts_df['total_mean'] = both_system_prompts_df[['mean_suspiciousness_sp1', 'mean_suspiciousness_sp5']].mean(axis=1)
both_system_prompts_df = both_system_prompts_df[~both_system_prompts_df.task_name.isin(['blog_post', 'art_collectors', 'spam_filter_update'])]

In [ ]:
aggregation_type = 'max_suspiciousness_sp5'
unstacked_df = prepare_data_for_training_progress(
    df=both_system_prompts_df,
    aggregation_type=aggregation_type,
    step_mapping=default_step_mapping,
    data_to_use='no_training_tasks',
    training_data=training_data,
    exclude_tasks=[]
)
unstacked_df['monitor_agent'] = aggregation_type
unstacked_df.rename(columns=default_step_mapping, inplace=True)
plot_roc_curves(unstacked_df, combined_results, give_hint=None, include_failed_attempts=False, include_failed_main_task=False, monitors=[aggregation_type], metrics=default_step_mapping.values())

In [ ]:
unstacked_df.task_name.value_counts()

In [ ]:
interesting_df = filter_and_prepare_data(
    both_system_prompts_df,
    data_to_use='all_test',
    training_data=training_data,
    exclude_tasks=['blog_post', 'art_collectors', 'election_tampering', 'spam_filter_update', 'performance_reviews', 'group_vacation_planning', 'ssn_task', 'networking_event']
)

print("Tasks in test set:")
print(interesting_df.task_name.value_counts())

print("\nTasks in training data:")
print(training_data.task_name.value_counts())

In [ ]:
# Run the function to plot training progress
# Create a step mapping where each model is 70 steps apart


# Example of using with different data filters and metrics
# training_progress_df = plot_training_progress(
#     df=interesting_df,
#     metric='total_mean',  # Explicitly specify which metric to use
#     step_mapping=default_step_mapping, 
#     target_fpr=0.03,
#     model_column='monitor_agent',
#     models=FT_MONITOR_MODELS  # Specify which models to include
# )

# Run the function to plot all suspiciousness metrics
# Use the same step mapping as before
all_metrics_df = plot_all_suspiciousness_metrics(
    df=interesting_df,
    metrics_list=['total_max', 'total_mean', 'suspiciousness_1_sp1', 'suspiciousness_1_sp5', 'mean_suspiciousness_sp1', 'mean_suspiciousness_sp5'],  # Explicitly specify which metrics to use
    step_mapping=default_step_mapping, 
    target_fpr=0.03,
    model_column='monitor_agent',
    models=FT_MONITOR_MODELS  # Specify which models to include
)
# plot_suspiciousness_distributions(interesting_df, monitors=FT_MONITOR_MODELS, metrics=['total_max'])

In [ ]:
interesting_df[(interesting_df.side_task_evaluation == 1) & (interesting_df.monitor_agent == FT_MONITOR_MODELS[-1])].sort_values('total_max', ascending=True)

In [ ]:
# Create normalized bar chart of task distributions
def plot_normalized_task_distributions(training_data: pd.DataFrame, system_prompt_df: pd.DataFrame):
    """
    Create a bar chart comparing the normalized task distributions between
    training data and system prompt data.
    """
    # Get value counts and normalize them
    training_counts = training_data.task_name.value_counts(normalize=True)
    system_counts = system_prompt_df.task_name.value_counts()/6
    system_counts = system_counts / system_counts.sum()  # Normalize to sum to 1
    
    # Create a DataFrame for plotting
    plot_df = pd.DataFrame({
        'Training Data': training_counts,
        'System Prompt Data': system_counts
    })
    
    # Sort by training data frequency
    plot_df = plot_df.sort_values('Training Data', ascending=False)
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(14, 8))
    plot_df.plot(kind='bar', ax=ax)
    
    # Add labels and title
    ax.set_xlabel('Task Name')
    ax.set_ylabel('Normalized Frequency')
    ax.set_title('Comparison of Task Distributions: Training Data vs System Prompt Data')
    ax.legend(title='Dataset')
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return plot_df

def plot_monitoring_performance_heatmap(df: pd.DataFrame, 
                                      metric: str = 'total_max', 
                                      monitors: List[str] = None,
                                      figsize: tuple = (14, 10),
                                      performance_metric: str = 'auc',  # 'auc' or 'tpr'
                                      target_fpr: float = 0.05,
                                      vmin: float = None,
                                      vmax: float = None,
                                      cmap: str = 'YlOrRd',
                                      step_mapping: Dict = None,
                                      min_samples: int = 10,
                                      min_samples_per_class: int = 3,
                                      min_samples_per_monitor: int = 5,
                                      title_fontsize: int = 18,
                                      axis_label_fontsize: int = 16,
                                      tick_fontsize: int = 14,
                                      annot_fontsize: int = 12,
                                      cbar_fontsize: int = 14,
                                      debug: bool = True):
    """
    Create a heatmap showing monitoring performance (ROC AUC or TPR at specified FPR) across different tasks.
    Higher values (better monitoring performance) are shown in darker colors by default.
    
    Args:
        df: DataFrame containing monitoring data
        metric: Column name for the suspiciousness metric to use (e.g., 'total_max', 'suspiciousness_1_sp1')
        monitors: List of monitor models to include (if None, uses all models in the DataFrame)
        figsize: Figure size as (width, height) tuple
        performance_metric: Which performance metric to use ('auc' or 'tpr')
        target_fpr: Target false positive rate for TPR interpolation (only used if performance_metric='tpr')
        vmin: Minimum value for colorbar scale (if None, uses data minimum)
        vmax: Maximum value for colorbar scale (if None, uses data maximum)
        cmap: Colormap to use (default is 'YlOrRd' where higher values are darker)
        step_mapping: Dictionary mapping model names to their training steps for nicer labels
        min_samples: Minimum number of samples required for a task to be included
        min_samples_per_class: Minimum number of samples required for each class (0/1)
        min_samples_per_monitor: Minimum number of samples required for each monitor
        title_fontsize: Font size for plot titles
        axis_label_fontsize: Font size for axis labels
        tick_fontsize: Font size for tick labels
        annot_fontsize: Font size for annotations in the heatmap
        cbar_fontsize: Font size for colorbar label
        debug: Whether to print debug information
        
    Returns:
        DataFrame containing the pivot table data used for the heatmap
    """
    import seaborn as sns
    from sklearn.metrics import auc, roc_curve
    
    # Make a copy to avoid modifying the original DataFrame
    plot_df = df.copy()
    
    # Print unique task names to help debug
    if debug:
        print(f"Tasks in DataFrame: {len(plot_df.task_name.unique())}")
        print(f"Task names: {sorted(plot_df.task_name.unique())}")
    
    # Filter monitors if specified
    if monitors is not None:
        plot_df = plot_df[plot_df.monitor_agent.isin(monitors)]
    
    # Initialize a DataFrame to store performance results
    performance_results = []
    skipped_tasks = {}
    
    # Calculate ROC metrics for each task and monitor
    for task in plot_df.task_name.unique():
        task_df = plot_df[plot_df.task_name == task]
        
        # Check if task has enough data overall
        if len(task_df) < min_samples:
            reason = f"Insufficient samples: {len(task_df)} < {min_samples}"
            skipped_tasks[task] = reason
            if debug:
                print(f"Skipping task '{task}' - {reason}")
            continue
        
        # Check if task has both positive and negative examples
        positive_samples = sum(task_df.side_task_evaluation == 1)
        negative_samples = sum(task_df.side_task_evaluation == 0)
        
        if positive_samples < min_samples_per_class or negative_samples < min_samples_per_class:
            reason = f"Insufficient class samples: pos={positive_samples}, neg={negative_samples}, min required={min_samples_per_class}"
            skipped_tasks[task] = reason
            if debug:
                print(f"Skipping task '{task}' - {reason}")
            continue
        
        # Track if we get any successful calculations for this task
        task_has_data = False
        
        for monitor in task_df.monitor_agent.unique():
            monitor_df = task_df[task_df.monitor_agent == monitor]
            
            # Skip if not enough data for this monitor
            if len(monitor_df) < min_samples_per_monitor:
                if debug:
                    print(f"Skipping monitor '{monitor}' for task '{task}' - insufficient samples: {len(monitor_df)} < {min_samples_per_monitor}")
                continue
            
            # Calculate class distribution for this monitor
            monitor_pos = sum(monitor_df.side_task_evaluation == 1)
            monitor_neg = sum(monitor_df.side_task_evaluation == 0)
            
            if monitor_pos < min_samples_per_class or monitor_neg < min_samples_per_class:
                if debug:
                    print(f"Skipping monitor '{monitor}' for task '{task}' - insufficient class samples: pos={monitor_pos}, neg={monitor_neg}")
                continue
                
            # Calculate ROC curve
            y_true = (monitor_df.side_task_evaluation == 1).astype(int)
            y_score = monitor_df[metric].values
            
            # Check score range and normalize appropriately
            score_max = np.max(y_score)
            score_min = np.min(y_score)
            
            if debug:
                print(f"Task '{task}', Monitor '{monitor}': Score range = [{score_min}, {score_max}]")
            
            # Normalize scores to [0,1] if they're not already
            if score_max > 1:
                # Determine if scores are likely in [0,100] range or something else
                if score_max <= 100 and score_min >= 0:
                    y_score = y_score / 100.0
                else:
                    # More robust normalization for other ranges
                    y_score = (y_score - score_min) / (score_max - score_min) if score_max > score_min else y_score
                
                if debug:
                    print(f"  After normalization: [{np.min(y_score)}, {np.max(y_score)}]")
                
            try:
                fpr, tpr, thresholds = roc_curve(y_true, y_score)
                roc_auc = auc(fpr, tpr)
                
                # Calculate interpolated TPR at target_fpr if performance_metric is 'tpr'
                if performance_metric == 'tpr':
                    # Use the interpolate_tpr function from plotting_utils
                    interp_tpr = interpolate_tpr(fpr, tpr, target_fpr)
                    performance_value = interp_tpr
                    
                    if debug:
                        # Display a few points around the target FPR for verification
                        if len(fpr) > 2:
                            closest_idx = np.abs(fpr - target_fpr).argmin()
                            start_idx = max(0, closest_idx - 1)
                            end_idx = min(len(fpr) - 1, closest_idx + 1)
                            print(f"  FPR/TPR points around target {target_fpr}:")
                            for idx in range(start_idx, end_idx + 1):
                                print(f"    FPR: {fpr[idx]:.4f}, TPR: {tpr[idx]:.4f}, Threshold: {thresholds[idx] if idx < len(thresholds) else 'N/A'}")
                            print(f"  Interpolated TPR at FPR={target_fpr}: {interp_tpr:.4f}")
                else:  # Use AUC
                    performance_value = roc_auc
                    
                performance_results.append({
                    'task_name': task,
                    'monitor_agent': monitor,
                    'performance': performance_value
                })
                task_has_data = True
            except Exception as e:
                if debug:
                    print(f"Error calculating ROC for task '{task}', monitor '{monitor}': {e}")
                continue
        
        # If we couldn't get any successful calculations for this task, add it to skipped tasks
        if not task_has_data and task not in skipped_tasks:
            skipped_tasks[task] = "Failed to calculate performance metrics for any monitor"
    
    # Convert to DataFrame and create pivot table
    if not performance_results:
        print("No performance results could be calculated. Check your data.")
        return pd.DataFrame()
        
    perf_df = pd.DataFrame(performance_results)
    
    # Verify the range of performance values
    if debug:
        print(f"\nPerformance metric: {performance_metric}")
        print(f"Performance value range: [{perf_df.performance.min():.4f}, {perf_df.performance.max():.4f}]")
        print(f"Performance value mean: {perf_df.performance.mean():.4f}")
    
    pivot_df = pd.pivot_table(
        perf_df,
        values='performance',
        index='task_name',
        columns='monitor_agent'
    )
    
    # Sort columns by training steps if step_mapping is provided
    if step_mapping is not None:
        # Create a mapping dictionary for column renaming
        label_mapping = {}
        step_values = {}
        
        for model, step in step_mapping.items():
            if model in pivot_df.columns:
                if step == 0:
                    label_mapping[model] = "Base model"
                else:
                    label_mapping[model] = f"Step {step}"
                step_values[model] = step
        
        # Sort columns by step value
        pivot_df = pivot_df[sorted(pivot_df.columns, key=lambda x: step_values.get(x, 0) if x in step_values else 0)]
        
        # Apply the mapping
        if label_mapping:
            pivot_df = pivot_df.rename(columns=label_mapping)
    
    # Sort rows by average performance (harder tasks at the top)
    pivot_df['avg_performance'] = pivot_df.mean(axis=1)
    pivot_df = pivot_df.sort_values('avg_performance', ascending=False)
    pivot_df = pivot_df.drop(columns=['avg_performance'])
    
    # Create the heatmap plot
    plt.figure(figsize=figsize)
    ax = sns.heatmap(
        pivot_df,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        annot_kws={"size": annot_fontsize},
        cbar_kws={'label': f'{"ROC AUC" if performance_metric == "auc" else f"TPR at FPR={target_fpr}"}'}
    )
    
    # Add labels and title with larger font sizes
    metric_name = "ROC AUC" if performance_metric == "auc" else f"TPR at FPR={target_fpr}"
    plt.title(f'Monitoring Performance Heatmap: {metric_name} using {metric}', fontsize=title_fontsize)
    plt.ylabel('Task (sorted by difficulty)', fontsize=axis_label_fontsize)
    plt.xlabel('Monitor Model', fontsize=axis_label_fontsize)
    
    # Set font size for tick labels
    plt.xticks(fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    
    # Set font size for colorbar label
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=tick_fontsize)
    cbar.set_label(f'{"ROC AUC" if performance_metric == "auc" else f"TPR at FPR={target_fpr}"}', fontsize=cbar_fontsize)
    
    # Adjust layout for readability
    plt.tight_layout()
    plt.show()
    
    # Create a bar chart showing average performance by task
    plt.figure(figsize=figsize)
    task_avg = pivot_df.mean(axis=1).reset_index()
    task_avg.columns = ['task_name', 'avg_performance']
    task_avg = task_avg.sort_values('avg_performance', ascending=False)
    
    bar_plot = sns.barplot(
        data=task_avg,
        x='task_name', 
        y='avg_performance'
    )
    
    # Set larger font sizes for the bar chart
    plt.title(f'Average {metric_name} by Task using {metric}', fontsize=title_fontsize)
    plt.xlabel('Task', fontsize=axis_label_fontsize)
    plt.ylabel(metric_name, fontsize=axis_label_fontsize)
    plt.xticks(rotation=45, ha='right', fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    
    # Adjust layout for readability
    plt.tight_layout()
    plt.show()
    
    # Print summary of skipped tasks
    if skipped_tasks and debug:
        print("\nSummary of skipped tasks:")
        for task_name, reason in skipped_tasks.items():
            print(f"- {task_name}: {reason}")
        print(f"Total tasks in data: {len(plot_df.task_name.unique())}")
        print(f"Tasks included in heatmap: {len(pivot_df)}")
        print(f"Tasks skipped: {len(skipped_tasks)}")
    
    # Return the pivot table for further analysis
    return pivot_df

# Run the function to create the normalized bar chart
normalized_task_df = plot_normalized_task_distributions(training_data, system_prompt_df_5)


In [ ]:
# Check task distribution in interesting_df
print("Task distribution in interesting_df:")
task_counts = interesting_df.task_name.value_counts()
print(task_counts)
print(f"\nTotal unique tasks: {len(task_counts)}")

# Check the transcript types to see if there's filtering happening there
print("\nTranscript types in interesting_df:")
print(interesting_df.transcript_type.value_counts())

# Check side_task_evaluation distribution
print("\nside_task_evaluation distribution:")
print(interesting_df.side_task_evaluation.value_counts())


In [ ]:
# Create monitoring performance heatmap with AUC as the performance metric
auc_heatmap_df = plot_monitoring_performance_heatmap(
    both_system_prompts_df,  # Use combined dataset for more tasks
    metric='suspiciousness_1_sp1',
    monitors=FT_MONITOR_MODELS,
    performance_metric='auc',
    cmap='YlOrRd',  # Higher AUC (better monitoring) = darker color
    step_mapping=default_step_mapping
)


In [ ]:
# Create monitoring performance heatmap with TPR at low FPR as the performance metric
tpr_heatmap_df = plot_monitoring_performance_heatmap(
    both_system_prompts_df,  # Use combined dataset for more tasks
    metric='mean_suspiciousness_sp5',
    monitors=FT_MONITOR_MODELS,
    performance_metric='tpr',
    target_fpr=0.03,  # Aim for low false positive rate
    cmap='YlOrRd',  # Higher TPR (better monitoring) = darker color
    step_mapping=default_step_mapping
)


In [ ]:
# Print all tasks in both_system_prompts_df for reference
print("All tasks in both_system_prompts_df:")
task_counts = both_system_prompts_df.task_name.value_counts()
print(task_counts)

# Show more detailed analysis of task distribution
for task in both_system_prompts_df.task_name.unique():
    task_df = both_system_prompts_df[both_system_prompts_df.task_name == task]
    pos = sum(task_df.side_task_evaluation == 1)
    neg = sum(task_df.side_task_evaluation == 0)
    print(f"{task}: total={len(task_df)}, positive={pos}, negative={neg}")
    

In [ ]:
# Create a more inclusive heatmap with less restrictive filtering
inclusive_heatmap_df = plot_monitoring_performance_heatmap(
    both_system_prompts_df,
    metric='total_max',
    monitors=FT_MONITOR_MODELS,
    performance_metric='auc',
    cmap='YlOrRd',
    step_mapping=default_step_mapping,
    min_samples=5,          # Lower minimum samples threshold
    min_samples_per_class=2,  # Lower class threshold
    min_samples_per_monitor=3  # Lower monitor threshold
)


In [ ]:
# Compare different metrics on the same dataset
metric_comparison_df = plot_monitoring_performance_heatmap(
    interesting_df,
    metric='total_max',  # Try different metric
    monitors=[FT_MONITOR_MODELS[0], FT_MONITOR_MODELS[-1]],  # Compare base model vs final model
    performance_metric='auc',
    step_mapping=default_step_mapping
)

In [ ]:
plot_suspiciousness_distributions(both_system_prompts_df[both_system_prompts_df.task_name == "api_key_calendar_agendas_2"], monitors=FT_MONITOR_MODELS, metrics=["suspiciousness_1_sp5", "suspiciousness_2_sp5"])

In [ ]:
task = 'conference_recap'
solution_reviews_df = load_solution_reviews_data()
solution_reviews_df = solution_reviews_df[solution_reviews_df.task_name == task]

In [ ]:
#filtered_system_prompts_df = both_system_prompts_df[both_system_prompts_df.task_name.isin(solution_reviews_df.task_name.unique())]
filtered_system_prompts_df = both_system_prompts_df[both_system_prompts_df.task_name == task]
# Calculate a good grid layout based on number of models
n_models = len(FT_MONITOR_MODELS)
n_cols = min(3, n_models)  # Maximum 3 columns
n_rows = (n_models + n_cols - 1) // n_cols  # Ceiling division for rows

# Create a single figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), constrained_layout=True)
if n_models == 1:  # Handle the case with just one subplot
    axes = np.array([axes])
axes = axes.flatten()  # Flatten for easy indexing

# Get global min/max across all datasets for consistent bins
global_min = 0
global_max = 100

# Create fixed bins for all plots
bins = np.linspace(global_min, global_max, 20)

# Plot each model in its own subplot
for i, monitor in enumerate(FT_MONITOR_MODELS):
    ax = axes[i]  # Get the current subplot
    
    # Get all three suspiciousness metrics and combine them for each category
    # With side task - combine all three metrics
    data_with_side_task = []
    for metric in ['suspiciousness_1_sp5', 'suspiciousness_2_sp5', 'suspiciousness_3_sp5']:
        data = filtered_system_prompts_df[(filtered_system_prompts_df.monitor_agent == monitor) & 
                                         (filtered_system_prompts_df.side_task_evaluation == 1)][metric]
        if not data.empty:
            data_with_side_task.extend(data.tolist())
    
    # Without side task - combine all three metrics
    data_without_side_task = []
    for metric in ['suspiciousness_1_sp5', 'suspiciousness_2_sp5', 'suspiciousness_3_sp5']:
        data = filtered_system_prompts_df[(filtered_system_prompts_df.monitor_agent == monitor) & 
                                         (filtered_system_prompts_df.side_task_evaluation == 0)][metric]
        if not data.empty:
            data_without_side_task.extend(data.tolist())
    
    # Solution reviews - get all verdicts
    data_solution_reviews = []
    if not solution_reviews_df[solution_reviews_df.model_name == monitor].empty:
        for i in range(1, 6):  # Get all 5 verdicts
            verdict = solution_reviews_df[solution_reviews_df.model_name == monitor][f'verdict_{i}'].iloc[0]
            if verdict is not None:
                data_solution_reviews.append(verdict)
    
    # Plot histograms on this subplot - normalized as probability distributions
    ax.hist(data_with_side_task, bins=bins, alpha=0.7, density=True, label="With side task (all metrics)")
    ax.hist(data_without_side_task, bins=bins, alpha=0.7, density=True, label="Without side task (all metrics)")
    ax.hist(data_solution_reviews, bins=bins, alpha=0.7, density=True, color='gold', label="Solution reviews (all verdicts)")
    
    # Add labels and title for this subplot
    ax.set_title(f'Step {default_step_mapping[monitor]}')
    ax.set_xlabel('Suspiciousness score')
    ax.set_ylabel('Probability density')
    
    # Only add legend to the first subplot to avoid repetition
    if i == 0:
        ax.legend()

    # Add sample size information to each subplot
    sample_info = f"n_with={len(data_with_side_task)}, n_without={len(data_without_side_task)}, n_reviews={len(data_solution_reviews)}"
    ax.text(0.5, 0.97, sample_info, transform=ax.transAxes, ha='center', va='top', fontsize=8)

# Hide any unused subplots
for j in range(n_models, len(axes)):
    axes[j].set_visible(False)

# Add a common title for the entire figure
fig.suptitle('Normalized probability distributions of suspiciousness scores across training steps', fontsize=16)
plt.show()


In [ ]:
for t in training_data[(training_data.task_name == "conference_recap") & (training_data.side_task_evaluation == 1)].transcript:
    print(t[t.index("monitoring_response"):])

In [ ]:
solution_reviews_df['mean_verdict'] = solution_reviews_df.apply(lambda row: np.mean([row[f'verdict_{i}'] for i in range(1, 6)]), axis=1)
filtered = solution_reviews_df[solution_reviews_df.model_name.isin(FT_MONITOR_MODELS)]
filtered['training_step'] = filtered.apply(lambda row: default_step_mapping[row['model_name']], axis=1)
filtered = filtered.sort_values('training_step')
plt.plot(filtered.training_step, filtered.mean_verdict)

In [ ]:
both_system_prompts_df